# Supervised Model Development

This notebook implements the supervised-learning workflow for predicting whether a banking complaint ends with any consumer monetary relief or no relief. The primary source file is `data/processed/consumer_banking_relief.parquet`. Due to size limitations, the data is not available on Github, however, please see the sample_data folder of the repo for a glimpse into 100 examples records.

AI Assistance:
OpenAI ChatGPT was used for code debugging, code generation, code organization,
and code methodological brainstorming. All final modeling, implementation,
validation, commentary, and interpretation were performed and verified by the authors.


In [2]:
# pip install -r ../requirements.txt


## Notebook Overview

In this notebook, we will

1. Load and quality-check the complaint data
2. Engineer leakage-aware structured features plus simple narrative indicators for relief-response prediction
3. Create a stratified train/test split for our model training
4. Tune and compare four model families: Logistic Regression, KNN, Random Forest, and Support Vector Machine
5. Select a final development model based on Cross Validation F1-score performance and save artifacts for downstream evaluation

The target is defined as `target_relief = 1` when `Company response to consumer` is either `Closed with monetary relief` or `Closed with non-monetary relief`, and `0` when the response is `Closed with explanation`. Because the filtered dataset contains only these two grouped outcomes, the existing binary classifiers remain appropriate.


## Table of Contents

1. [Data Loading and Exploration](#Data-Loading-and-Exploration)
2. [Feature Engineering](#Feature-Engineering)
3. [Train/Test Setup](#Train/Test-Setup)
4. [Model #1 Logistic Regression](#Model-#1-Logistic-Regression)
5. [Model #2 KNN](#Model-#2-KNN)
6. [Model #3 Random Forest](#Model-#3-Random-Forest)
7. [Model #4 Support Vector Machine](#Model-#4-Support-Vector-Machine)
8. [Final Model Selection](#Final-Model-Selection)


### Data Loading and Exploration

We start by validating that the consumer_banking_relief parquet is internally consistent enough for supervised learning. Before training any model, we check for duplicate complaint identifiers, target completeness, date parsing issues, negative date gaps, and the amount of missingness in high-value fields such as the narrative (which will be vital for later models developed using Unsupervised generated features)


In [3]:
import os
from pathlib import Path
import warnings

import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "consumer_banking_relief.parquet"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)
OUTPUT_TABLES_DIR = PROJECT_ROOT / "output_tables"
OUTPUT_TABLES_DIR.mkdir(exist_ok=True)

# Due to model training time, we'll use a subset of the data and fewer CV folds for quick testing. 
# These settings should be adjusted for a more thorough model development process.
MODEL_SCOPE = "all_banking"
QUICK_TEST_MODE = True
RANDOM_STATE = 42
CV_FOLDS = 3 if QUICK_TEST_MODE else 5
MAX_MODEL_ROWS = 30000 if QUICK_TEST_MODE else 150000
KNN_MAX_TRAIN_ROWS = 5000 if QUICK_TEST_MODE else 30000
RF_N_ESTIMATORS = 150 if QUICK_TEST_MODE else 300

# Limit the number of parallel jobs to avoid overloading the system during testing. Adjust as needed for full runs.
N_JOBS = min(2, max(1, (os.cpu_count() or 2) - 1))

assert DATA_PATH.exists(), f"Expected source parquet at {DATA_PATH}"

raw_df = pd.read_parquet(DATA_PATH)

print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns from {DATA_PATH.name}")
raw_df.head()


Loaded 1,123,191 rows and 18 columns from consumer_banking_relief.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,True,N/A,3477549
1,2019-12-20,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,FL,33064,NaN,N/A,Referral,2019-12-23,Closed with explanation,True,N/A,3475858
2,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136
3,2020-06-05,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,10466,NaN,Consent not provided,Web,2020-06-05,Closed with explanation,True,N/A,3684669
4,2024-01-16,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Add-on products and services,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,TX,76179,NaN,Consent not provided,Web,2024-01-16,Closed with monetary relief,True,N/A,8161600


In [4]:
quality_df = raw_df.copy()
quality_df["Date received"] = pd.to_datetime(quality_df["Date received"], errors="coerce")
quality_df["Date sent to company"] = pd.to_datetime(quality_df["Date sent to company"], errors="coerce")
quality_df["company_lag_days"] = (
    quality_df["Date sent to company"] - quality_df["Date received"]
).dt.days

# quality summary table with dtype, missing count and percentage, and number of unique values for each column
quality_summary = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing_count": raw_df.isna().sum(),
        "missing_pct": (raw_df.isna().mean() * 100).round(2),
        "n_unique": raw_df.nunique(dropna=True),
    }
).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

# data quality checks summary, looks like the data is pretty clean. 
# Really just the narrative with missing values, which we already know about from our EDA.
data_quality_checks = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "duplicate_rows": int(raw_df.duplicated().sum()),
        "duplicate_complaint_ids": int(raw_df["Complaint ID"].duplicated().sum()),
        "null_target_count": int(raw_df["Company response to consumer"].isna().sum()),
        "unparseable_date_received": int(quality_df["Date received"].isna().sum()),
        "unparseable_date_sent": int(quality_df["Date sent to company"].isna().sum()),
        "negative_lag_rows": int((quality_df["company_lag_days"] < 0).sum()),
        "narrative_missing_pct": round(raw_df["Consumer complaint narrative"].isna().mean() * 100, 2),
        "target_relief_pct": round(raw_df["Company response to consumer"].isin(["Closed with monetary relief"]).mean() * 100, 3),
    },
    name="value",
)

display(data_quality_checks.to_frame())
display(quality_summary.head(15))
display(quality_df["company_lag_days"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)


,value
rows,1123191.000
columns,18.000
duplicate_rows,0.000
duplicate_complaint_ids,0.000
null_target_count,0.000
unparseable_date_received,0.000
unparseable_date_sent,0.000
negative_lag_rows,0.000
narrative_missing_pct,51.870
target_relief_pct,12.461


,dtype,missing_count,missing_pct,n_unique
Tags,str,923625,82.23,3
Consumer complaint narrative,str,582633,51.87,522407
Company public response,str,560643,49.92,11
Sub-issue,str,263776,23.48,130
Consumer consent provided?,str,41767,3.72,5
Sub-product,str,28253,2.52,28
State,str,18963,1.69,63
ZIP code,str,13842,1.23,25510
Complaint ID,int64,0,0.00,1123191
Date received,object,0,0.00,3800


,count,mean,std,min,1%,5%,50%,95%,99%,max
company_lag_days,1123191.0,1.666331,7.450357,0.0,0.0,0.0,0.0,8.0,36.0,543.0


Looking at the above, we can see that we don't have much to worry about from our quality checks. We already knew that the narrative portion was missing in roughly half of our cases. The summary df is interesting as it provides some good feature context. Based on the results, we won't want the Tags variable as it's unlikely to posess any signal. Furthermore, after discussion on some of the other items, we will be running another notebook that runs on the final data which will be futher refined to remove some null features for items that may be important such as sub-issue or sub-product. However, for practical purposes, we'll probably end up running a simple imputer to flag entries as missing so that our model stays useful for broader sets of complaints assuming enough signal is found in the remaining features. We'll see how egregious our results for the Failure analysis are later.


In [5]:
product_summary = (
    raw_df.groupby("Product", dropna=False)
    .agg(
        complaints=("Complaint ID", "count"),
        relief_rate=("Company response to consumer", lambda s: s.isin(["Closed with monetary relief", "Closed with non-monetary relief"]).mean()),
        narrative_available=("Consumer complaint narrative", lambda s: s.notna().mean()),
    )
    .sort_values("complaints", ascending=False)
)
product_summary[["relief_rate", "narrative_available"]] = (
    product_summary[["relief_rate", "narrative_available"]] * 100
).round(2)

display(product_summary)
display(raw_df["Company response to consumer"].value_counts(dropna=False).rename("count").to_frame())

high_null_columns = quality_summary.loc[quality_summary["missing_pct"] >= 20, ["missing_pct", "n_unique"]]
display(high_null_columns)


,complaints,relief_rate,narrative_available
Product,,,
Checking or savings account,364392,20.22,48.97
Mortgage,273650,6.26,47.72
Credit card,250118,34.28,45.03
Credit card or prepaid card,206232,29.35,52.66
Bank account or service,28799,28.01,35.85


,count
Company response to consumer,
Closed with explanation,876350
Closed with monetary relief,139957
Closed with non-monetary relief,105178
Closed,1706


,missing_pct,n_unique
Tags,82.23,3
Consumer complaint narrative,51.87,522407
Company public response,49.92,11
Sub-issue,23.48,130


Overall, the initial quality check shows a few important findings that shape the rest of the notebook:

- The processed banking dataset is large enough for supervised learning at over 200k records even with our filtered down subset.
- `Complaint ID` appears unique and the date fields parse cleanly, which reduces the need for dedup logic or date parsing.
- The filtered relief-response dataset is much more balanced than the untimely response dataset, so we use explicit imbalance handling and tractability controls in this notebook because the relief dataset is larger and may still benefit from rebalancing depending on the final class mix.
- Narrative coverage is still meaningful but incomplete, which motivates combining simple narrative indicators with structured complaint context.
- Missingness is concentrated in fields like `Tags` and `Company public response`, which make them unlikely to be useful. Furthermore, `Consumer complaint narrative` and `Sub-issue` also contain missing values, so our preprocessing needs robust imputers and careful leakage handling.

The product summary also confirms that this is a banking problem rather than a single product classification task. That variety is one reason we preserve `Product`, `Issue`, channel, geography, and company context as predictive inputs.


### Feature Engineering

We intentionally exclude fields that are likely to leak post-submission or post-resolution information into the prediction task. In particular, `Date sent to company`, `Timely response?`, `Company public response`, and `Consumer disputed?` are better suited for diagnostics than for prediction of the company monetary/non-monetary response outcome.

Our feature design tries to balance predictive power with practicality. If a feature is only known after the complaint has already been processed by the company, we exclude it from model training even if it would make prediction easier.

Narrative information in this notebook is intentionally kept simple:

- whether a narrative is present
- basic narrative length measures such as character and word count

More advanced narrative modeling is intentionally left out of this notebook so it does not overlap with the separate unsupervised clustering and feature generation workstream.


In [6]:
def make_one_hot_encoder(dense=False):
    kwargs = {"handle_unknown": "ignore"}
    try:
        return OneHotEncoder(sparse_output=not dense, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=not dense, **kwargs)

# The score estimator func lets us also evaluate on the holdout test set, as our cross validation scores will come from the GridSearchCV results
def score_estimator(estimator, X_test, y_test):
    if hasattr(estimator, "predict_proba"):
        y_score = estimator.predict_proba(X_test)[:, 1]
    else:
        y_score = estimator.decision_function(X_test)
    y_pred = estimator.predict(X_test)
    return {
        "holdout_accuracy": accuracy_score(y_test, y_pred),
        "holdout_average_precision": average_precision_score(y_test, y_score),
        "holdout_roc_auc": roc_auc_score(y_test, y_score),
        "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
        "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
        "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
        "holdout_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "y_score": y_score,
        "y_pred": y_pred,
    }

# Compile the results from the GridSearchCV and holdout test evaluation into a single row for easy comparison across models and hyperparameters
def build_results_row(model_name, search, X_test, y_test):
    best_idx = search.best_index_
    metric_names = [
        "accuracy",
        "average_precision",
        "roc_auc",
        "recall",
        "precision",
        "f1",
        "balanced_accuracy",
    ]
    row = {
        "model_family": model_name,
        "best_params": search.best_params_,
    }
    # Bulding out the row with the CV mean and std for each metric, using the best index from the GridSearchCV results
    for metric in metric_names:
        row[f"cv_mean_{metric}"] = search.cv_results_[f"mean_test_{metric}"][best_idx]
        row[f"cv_std_{metric}"] = search.cv_results_[f"std_test_{metric}"][best_idx]
    
    # Then we evaluate the best estimator on the holdout test set and add those scores to the row as well
    holdout_scores = score_estimator(search.best_estimator_, X_test, y_test)
    row.update({k: v for k, v in holdout_scores.items() if not k.startswith("y_")})
    return row, holdout_scores

# As mentioned above, we'll remove any columns that might contain info not available for us to use at the time of prediction
# Although we looked at Timely response before, because our target is now monetary relief, it might be more likely to contain info about the outcome of the complaint, so we'll exclude it from modeling as well.
leakage_columns = [
    "Timely response?",
    "Complaint ID",
    "Date sent to company",
    "Company response to consumer",
    "Company public response",
    "Consumer disputed?",
    "Consumer consent provided?",
]

model_df = raw_df.copy()
model_df["Date received"] = pd.to_datetime(model_df["Date received"], errors="coerce")

# Setting up our target for monetary relief, which is 1 if the company response to consumer is "Closed with monetary relief" and 0 otherwise
model_df["target_relief"] = model_df["Company response to consumer"].isin(["Closed with monetary relief"]).astype(int)

text_series = model_df["Consumer complaint narrative"].fillna("")
model_df["narrative_present"] = text_series.str.len().gt(0).astype(int)
model_df["narrative_char_count"] = text_series.str.len()
model_df["narrative_word_count"] = text_series.str.split().str.len().fillna(0)

model_df["received_year"] = model_df["Date received"].dt.year
model_df["received_month"] = model_df["Date received"].dt.month
model_df["received_quarter"] = model_df["Date received"].dt.quarter
model_df["received_dayofweek"] = model_df["Date received"].dt.dayofweek
model_df["received_day"] = model_df["Date received"].dt.day
model_df["received_days_since_start"] = (
    model_df["Date received"] - model_df["Date received"].min()
).dt.days
model_df["zip3"] = (
    model_df["ZIP code"].fillna("").astype(str).str.extract(r"(\d{3})", expand=False).fillna("missing")
)

top_category_limits = {
    "Company": 40 if QUICK_TEST_MODE else 75,
    "Sub-product": 20 if QUICK_TEST_MODE else 30,
    "Issue": 25 if QUICK_TEST_MODE else 40,
    "Sub-issue": 35 if QUICK_TEST_MODE else 60,
    "State": 20 if QUICK_TEST_MODE else 30,
    "zip3": 50 if QUICK_TEST_MODE else 100,
}
for col, top_n in top_category_limits.items():
    values = model_df[col].fillna("missing").astype(str)
    keep = set(values.value_counts().head(top_n).index)
    model_df[col] = values.where(values.isin(keep), other="__OTHER__")
model_df["Submitted via"] = model_df["Submitted via"].fillna("missing").astype(str)
model_df["Product"] = model_df["Product"].fillna("missing").astype(str)

model_df = model_df.dropna(subset=["Date received", "target_relief"]).copy()
if len(model_df) > MAX_MODEL_ROWS:
    _, model_df = train_test_split(
        model_df,
        test_size=MAX_MODEL_ROWS,
        stratify=model_df["target_relief"],
        random_state=RANDOM_STATE,
    )

print(f"Modeling rows after optional tractability down-sampling: {len(model_df):,}")
print(f"Relief class rate: {model_df['target_relief'].mean() * 100:.3f}%")
display(pd.Series(leakage_columns, name="excluded_from_modeling"))


Modeling rows after optional tractability down-sampling: 30,000
Relief class rate: 12.460%


0                Timely response?
1                    Complaint ID
2            Date sent to company
3    Company response to consumer
4         Company public response
5              Consumer disputed?
6      Consumer consent provided?
Name: excluded_from_modeling, dtype: str

A few design choices here:

- We keep only lightweight narrative indicators in this notebook, specifically whether a narrative exists and how long it is. Again, this is so that we can compare to when we include Unsupervised features.
- We convert the prediction target to `target_relief`, making relief as the positive class for a straightforward binary setup. This lets us focus in on our problem statement which is to predict whether a monetary response is possible or not, rather than deal with the field's general multi-class output.
- We include simple calendar features from `Date received` because operational timing effects can matter, but we exclude `Date sent to company` because it is too close to the response workflow and could leak downstream process information.
- We use the full filtered dataset rather than applying balancing-oriented upsampling or downsampling, since the underlying class balance can be checked directly, but we also retain explicit imbalance handling for this larger relief dataset.


### Train/Test Setup

We use a stratified split and 5-fold stratified cross-validation and report accuracy, precision, recall, F1, ROC AUC, balanced accuracy, and average precision for each model. Cross-fold model selection still uses F1, but we use stratified splits plus targeted resampling for the distance-based model because this relief dataset is larger and the combined positive class may still benefit from balancing support.


In [7]:
target_col = "target_relief"

structured_numeric = [
    "received_year",
    "received_month",
    "received_quarter",
    "received_dayofweek",
    "received_day",
    "received_days_since_start",
    "narrative_present",
    "narrative_char_count",
    "narrative_word_count",
]

logistic_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "zip3",
]

knn_categorical = ["Product", "Issue", "State", "Submitted via"]
rf_categorical = ["Product", "Sub-product", "Issue", "Sub-issue", "State", "Submitted via", "zip3"]

logistic_features = logistic_categorical + structured_numeric
knn_features = knn_categorical + structured_numeric
rf_features = rf_categorical + structured_numeric

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)

scoring = {
    "accuracy": "accuracy",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train relief rate: {train_df[target_col].mean() * 100:.3f}%")
print(f"Test relief rate: {test_df[target_col].mean() * 100:.3f}%")
print(f"Grid-search workers: {N_JOBS}")
print(f"Quick test mode: {QUICK_TEST_MODE}")
print(f"Max modeling rows: {MAX_MODEL_ROWS}")


Train rows: 24,000
Test rows: 6,000
Train relief rate: 12.458%
Test relief rate: 12.467%
Grid-search workers: 2
Quick test mode: True
Max modeling rows: 30000


### Model #1 Logistic Regression

This linear baseline is a strong fit for one-hot encoded categorical features plus the simple narrative indicators. We tune regularization strength with class balancing because the larger relief dataset can still benefit from more stable minority-class treatment.

Design expectation: Logistic Regression should remain a strong and interpretable baseline because it handles high-cardinality categorical features well and produces stable decision boundaries.


In [13]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            logistic_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=300 if QUICK_TEST_MODE else 1000,
                solver="liblinear" if QUICK_TEST_MODE else "saga",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_grid = {
    "model__C": [0.5] if QUICK_TEST_MODE else [0.5, 1.0, 2.0],
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_log = train_df[logistic_features]
y_train_log = train_df[target_col]
X_test_log = test_df[logistic_features]
y_test_log = test_df[target_col]

logistic_search.fit(X_train_log, y_train_log)
logistic_row, logistic_scores = build_results_row("Logistic Regression", logistic_search, X_test_log, y_test_log)

logistic_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
logistic_holdout["model_family"] = "Logistic Regression"
logistic_holdout["score"] = logistic_scores["y_score"]
logistic_holdout["prediction"] = logistic_scores["y_pred"]

display(pd.DataFrame([logistic_row]).T)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


,0
model_family,Logistic Regression
best_params,{'model__C': 0.5}
cv_mean_accuracy,0.625875
cv_std_accuracy,0.004168
cv_mean_average_precision,0.278101
cv_std_average_precision,0.008977
cv_mean_roc_auc,0.757511
cv_std_roc_auc,0.004732
cv_mean_recall,0.780267
cv_std_recall,0.003634


### Model #2 KNN

KNN gives us a very different, instance-based learning family. Because distance-based models scale poorly in very high dimensions, this version uses compact structured features plus simple narrative indicators rather than raw text. We also include SMOTE inside the KNN pipeline for the relief task.

Design expectation: KNN is less likely to dominate on this task, but it provides a useful contrast because it relies on local neighborhood structure rather than a global linear or tree-based decision rule.


In [14]:
if KNN_MAX_TRAIN_ROWS is not None and len(train_df) > KNN_MAX_TRAIN_ROWS:
    _, knn_train_df = train_test_split(
        train_df,
        test_size=KNN_MAX_TRAIN_ROWS,
        stratify=train_df[target_col],
        random_state=RANDOM_STATE,
    )
else:
    knn_train_df = train_df.copy()

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=True)),
                ]
            ),
            knn_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

knn_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", knn_preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.50, k_neighbors=3)),
        ("model", KNeighborsClassifier()),
    ]
)

knn_grid = {
    "smote__sampling_strategy": [0.50] if QUICK_TEST_MODE else [0.35, 0.50],
    "smote__k_neighbors": [3] if QUICK_TEST_MODE else [3, 5],
    "model__n_neighbors": [25] if QUICK_TEST_MODE else [15, 35, 75],
    "model__weights": ["distance"] if QUICK_TEST_MODE else ["uniform", "distance"],
    "model__p": [2] if QUICK_TEST_MODE else [1, 2],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_knn = knn_train_df[knn_features]
y_train_knn = knn_train_df[target_col]
X_test_knn = test_df[knn_features]
y_test_knn = test_df[target_col]

knn_search.fit(X_train_knn, y_train_knn)
knn_row, knn_scores = build_results_row("KNN", knn_search, X_test_knn, y_test_knn)

knn_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
knn_holdout["model_family"] = "KNN"
knn_holdout["score"] = knn_scores["y_score"]
knn_holdout["prediction"] = knn_scores["y_pred"]

display(pd.DataFrame([knn_row]).T)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


,0
model_family,KNN
best_params,"{'model__n_neighbors': 25, 'model__p': 2, 'mod..."
cv_mean_accuracy,0.680002
cv_std_accuracy,0.00918
cv_mean_average_precision,0.184484
cv_std_average_precision,0.004349
cv_mean_roc_auc,0.641635
cv_std_roc_auc,0.018515
cv_mean_recall,0.444684
cv_std_recall,0.027789


### Model #3 Random Forest

Random Forest gives us a tree-based, nonlinear baseline. It can capture interactions among the engineered structured and narrative-summary features without assuming linear relationships, and it can do so while also using class-balanced fitting on this larger relief target.

Design expectation: Random Forest can capture useful nonlinear interactions among issue type, submission channel, timing, and simple narrative indicators, but it may trade off some interpretability relative to the linear baselines.


In [15]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            rf_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            structured_numeric,
        ),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
        ),
    ]
)

rf_grid = {
    "model__max_depth": [12] if QUICK_TEST_MODE else [None, 12, 20],
    "model__min_samples_leaf": [10] if QUICK_TEST_MODE else [1, 5, 20],
    "model__max_features": ["sqrt"] if QUICK_TEST_MODE else ["sqrt", 0.5],
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_rf = train_df[rf_features]
y_train_rf = train_df[target_col]
X_test_rf = test_df[rf_features]
y_test_rf = test_df[target_col]

rf_search.fit(X_train_rf, y_train_rf)
rf_row, rf_scores = build_results_row("Random Forest", rf_search, X_test_rf, y_test_rf)

rf_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
rf_holdout["model_family"] = "Random Forest"
rf_holdout["score"] = rf_scores["y_score"]
rf_holdout["prediction"] = rf_scores["y_pred"]

display(pd.DataFrame([rf_row]).T)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


,0
model_family,Random Forest
best_params,"{'model__max_depth': 12, 'model__max_features'..."
cv_mean_accuracy,0.613
cv_std_accuracy,0.012642
cv_mean_average_precision,0.275351
cv_std_average_precision,0.009499
cv_mean_roc_auc,0.758804
cv_std_roc_auc,0.007224
cv_mean_recall,0.810704
cv_std_recall,0.024779


### Model #4 Support Vector Machine

Support Vector Machines add a margin-based model family that is distinct from both probabilistic linear models and tree-based methods. We use `LinearSVC` rather than a kernel SVM because the feature space is still fairly wide after one-hot encoding the categorical variables.

Design expectation: if the decision boundary is mostly linear but benefits from maximum-margin separation rather than calibrated probabilities, the SVM may perform competitively on F1 and recall.


In [16]:
svm_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "zip3",
]
svm_features = svm_categorical + structured_numeric

svm_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            svm_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", svm_preprocessor),
        ("model", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, dual="auto", max_iter=5000)),
    ]
)

svm_grid = {
    "model__C": [0.25] if QUICK_TEST_MODE else [0.25, 0.5, 1.0],
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_svm = train_df[svm_features]
y_train_svm = train_df[target_col]
X_test_svm = test_df[svm_features]
y_test_svm = test_df[target_col]

svm_search.fit(X_train_svm, y_train_svm)
svm_row, svm_scores = build_results_row("Support Vector Machine", svm_search, X_test_svm, y_test_svm)

svm_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", "narrative_present", target_col]].copy()
svm_holdout["model_family"] = "Support Vector Machine"
svm_holdout["score"] = svm_scores["y_score"]
svm_holdout["prediction"] = svm_scores["y_pred"]

display(pd.DataFrame([svm_row]).T)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


,0
model_family,Support Vector Machine
best_params,{'model__C': 0.25}
cv_mean_accuracy,0.61925
cv_std_accuracy,0.00572
cv_mean_average_precision,0.277261
cv_std_average_precision,0.009911
cv_mean_roc_auc,0.757343
cv_std_roc_auc,0.004917
cv_mean_recall,0.789297
cv_std_recall,0.002234


### Final Model Selection

We compare the best tuned model from each family using mean and standard deviation across cross-validation folds, then confirm the selected model on the holdout split.

Interpretation guidance: because the positive class is rare, the notebook now uses cross-validated F1 as the model-selection criterion. We still report accuracy, precision, recall, ROC AUC, balanced accuracy, and average precision so you can discuss the tradeoffs in the evaluation notebook.


In [17]:
results_df = pd.DataFrame([logistic_row, knn_row, rf_row, svm_row]).sort_values(
    by="cv_mean_f1", ascending=False
).reset_index(drop=True)

comparison_columns = [
    "model_family",
    "cv_mean_accuracy",
    "cv_std_accuracy",
    "cv_mean_average_precision",
    "cv_std_average_precision",
    "cv_mean_roc_auc",
    "cv_std_roc_auc",
    "cv_mean_recall",
    "cv_std_recall",
    "cv_mean_precision",
    "cv_std_precision",
    "cv_mean_f1",
    "cv_std_f1",
    "holdout_accuracy",
    "holdout_average_precision",
    "holdout_roc_auc",
    "holdout_recall",
    "holdout_precision",
    "holdout_f1",
]
display(results_df[comparison_columns])

searches = {
    "Logistic Regression": logistic_search,
    "KNN": knn_search,
    "Random Forest": rf_search,
    "Support Vector Machine": svm_search,
}
holdout_frames = {
    "Logistic Regression": logistic_holdout,
    "KNN": knn_holdout,
    "Random Forest": rf_holdout,
    "Support Vector Machine": svm_holdout,
}

best_model_name = results_df.loc[0, "model_family"]
best_search = searches[best_model_name]
best_holdout = holdout_frames[best_model_name].sort_values("score", ascending=False).reset_index(drop=True)

print(f"Selected development model: {best_model_name}")
print(f"Best params: {best_search.best_params_}")

comparison_path = OUTPUT_TABLES_DIR / "relief_supervised_model_comparison.csv"
visuals_comparison_path = OUTPUT_TABLES_DIR / "03_relief_model_comparison.csv"
holdout_path = ARTIFACT_DIR / "relief_best_model_holdout_predictions.parquet"
model_path = ARTIFACT_DIR / "best_relief_response_model.joblib"

results_df.to_csv(comparison_path, index=False)
results_df[comparison_columns].round(4).to_csv(visuals_comparison_path, index=False)
best_holdout.to_parquet(holdout_path, index=False)
joblib.dump(best_search.best_estimator_, model_path)

print(f"Saved comparison table to {comparison_path}")
print(f"Saved GitHub-friendly comparison table to {visuals_comparison_path}")
print(f"Saved holdout predictions to {holdout_path}")
print(f"Saved trained model to {model_path}")

best_holdout.head(10)


,model_family,cv_mean_accuracy,cv_std_accuracy,cv_mean_average_precision,cv_std_average_precision,cv_mean_roc_auc,cv_std_roc_auc,cv_mean_recall,cv_std_recall,cv_mean_precision,cv_std_precision,cv_mean_f1,cv_std_f1,holdout_accuracy,holdout_average_precision,holdout_roc_auc,holdout_recall,holdout_precision,holdout_f1
0,Random Forest,0.613000,0.012642,0.275351,0.009499,0.758804,0.007224,0.810704,0.024779,0.217575,0.004413,0.342993,0.005801,0.596667,0.292205,0.757446,0.842246,0.214870,0.342391
1,Logistic Regression,0.625875,0.004168,0.278101,0.008977,0.757511,0.004732,0.780267,0.003634,0.218978,0.002597,0.341976,0.003471,0.616833,0.298611,0.757433,0.804813,0.218512,0.343705
2,Support Vector Machine,0.619250,0.005720,0.277261,0.009911,0.757343,0.004917,0.789297,0.002234,0.217187,0.003165,0.340634,0.004097,0.610167,0.297084,0.756879,0.819519,0.217607,0.343899
3,KNN,0.680002,0.009180,0.184484,0.004349,0.641635,0.018515,0.444684,0.027789,0.180980,0.010844,0.257227,0.015381,0.694000,0.189858,0.652169,0.458556,0.193348,0.272006


Selected development model: Random Forest
Best params: {'model__max_depth': 12, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10}
Saved comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\relief_supervised_model_comparison.csv
Saved GitHub-friendly comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\visuals\03_relief_model_comparison.csv
Saved holdout predictions to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\relief_best_model_holdout_predictions.parquet
Saved trained model to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\best_relief_response_model.joblib


,Complaint ID,Product,Issue,Company,narrative_present,target_relief,model_family,score,prediction
0,15880652,Credit card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.787771,1
1,12996793,Credit card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.785076,1
2,5705958,Credit card or prepaid card,Fees or interest,"CITIBANK, N.A.",0,0,Random Forest,0.781190,1
3,3052522,Credit card or prepaid card,Fees or interest,CAPITAL ONE FINANCIAL CORPORATION,1,1,Random Forest,0.780222,1
4,4208188,Credit card or prepaid card,Fees or interest,SYNCHRONY FINANCIAL,1,0,Random Forest,0.779570,1
5,4735292,Credit card or prepaid card,Fees or interest,WELLS FARGO & COMPANY,1,0,Random Forest,0.778828,1
6,2656400,Credit card or prepaid card,Fees or interest,"CITIBANK, N.A.",0,1,Random Forest,0.778031,1
7,3796679,Credit card or prepaid card,Fees or interest,WELLS FARGO & COMPANY,0,1,Random Forest,0.775164,1
8,3800783,Credit card or prepaid card,Fees or interest,SYNCHRONY FINANCIAL,1,1,Random Forest,0.774485,1
9,7238789,Credit card or prepaid card,Fees or interest,"Bread Financial Holdings, Inc.",1,1,Random Forest,0.773843,1


### Narrative Ideas For Future Work

This supervised notebook intentionally avoids more complex narrative engineering, but these would be strong candidates for a separate narrative-focused workstream:

- topic clusters or complaint themes from unsupervised text clustering
- sentence embeddings from transformer-based encoders
- sentiment, emotion, or urgency indicators
- keyword groups tied to servicing delays, escrow, fraud, fees, or payment-processing issues
- readability or complexity measures
- named entities such as company names, products, locations, or dollar amounts
- temporal drift in narrative themes over time
- cluster membership or distance-to-centroid features merged back into supervised models
